In [1]:
from datetime import date

import pandas as pd
from sqlalchemy import func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # applications
    TLS202_APPLN_TITLE,         # titles
    TLS203_APPLN_ABSTR,         # abstracts
    TLS206_PERSON,              # persons (applicants and inventors)
    TLS207_PERS_APPLN,          # person <-> application links
    TLS209_APPLN_IPC,           # IPC classification
    TLS224_APPLN_CPC,           # CPC classification
    TLS231_INPADOC_LEGAL_EVENT, # legal events
)

In [2]:
patstat = PatstatClient()          # default: env="TEST"; use PatstatClient(env="PROD") for the full database
db = patstat.orm()                 # SQLAlchemy session used by all ORM queries below

This client instance is currently configured to use a test dataset with reduced number of publications (~10K).
Use PatstatClient(env='PROD') to use the complete PATSTAT dataset (>140M publications).
Use PatstatClient(env='TEST') to use the test dataset and avoid displaying this warning



In [4]:
#CPC code of the electric vehicle charging system field
#first we select all the patents present in the databse having a cpc code among one of this cpc code below

EV_CPC_CODES = (
    "B60L 53/",
    "B60L 55/",
    #"H02J 7/",
    "Y02T 90/",
    #------------
    #"H01B",
    #"H01F",
    #"H01H",
    #"H02", 
    #"H02B",
    #"H02J",
    #"H02M",
)

In [5]:
def symbol_prefix_condition(symbol_column, codes):
    """OR-condition: the blank-stripped classification symbol starts with one of the codes."""
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


def query_by_cpc(*columns):
    """Applications with at least one CPC symbol in the AI list."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
        .join(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)  
        .join(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)      
        .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, EV_CPC_CODES))
    )    

# Base columns shown for every application
APPLN_COLUMNS = (
    TLS201_APPLN.appln_id,
    TLS201_APPLN.appln_auth,
    TLS201_APPLN.appln_nr,
    TLS201_APPLN.appln_kind,
    TLS201_APPLN.appln_filing_date,
    TLS201_APPLN.appln_filing_year,
    TLS201_APPLN.docdb_family_id,
    TLS224_APPLN_CPC.cpc_class_symbol,
    TLS202_APPLN_TITLE.appln_title,
    TLS203_APPLN_ABSTR.appln_abstract
    
)    

In [6]:
#cpc search
q_cpc = query_by_cpc(*APPLN_COLUMNS).distinct()

df_cpc = patstat.df(q_cpc)
df_cpc.head(40)

,appln_id,appln_auth,appln_nr,appln_kind,appln_filing_date,appln_filing_year,docdb_family_id,cpc_class_symbol,appln_title,appln_abstract
0,50358722,US,3758836D,A,1972-02-03,1972,22834888,Y02T 90/12,ELECTRIC CONTROL SYSTEM FOR ALTERNATING CURREN...,When an alternating current motor is supplied ...
1,50358722,US,3758836D,A,1972-02-03,1972,22834888,Y02T 90/14,ELECTRIC CONTROL SYSTEM FOR ALTERNATING CURREN...,When an alternating current motor is supplied ...
2,53746527,US,83979977,A,1977-10-06,1977,25280653,Y02T 90/16,Automobile with wind driven generator,An automobile including a body provided with a...
3,49511120,US,31139281,A,1981-10-14,1981,23206681,B60L 53/00,Electric power generator and system,A generator adapted to be driven by rotational...
4,49511120,US,31139281,A,1981-10-14,1981,23206681,B60L 53/57,Electric power generator and system,A generator adapted to be driven by rotational...
5,49511120,US,31139281,A,1981-10-14,1981,23206681,Y02T 90/12,Electric power generator and system,A generator adapted to be driven by rotational...
6,49511120,US,31139281,A,1981-10-14,1981,23206681,Y02T 90/14,Electric power generator and system,A generator adapted to be driven by rotational...
7,50662501,US,40942082,A,1982-08-19,1982,23620416,B60L 53/30,Solar powered vehicle,A solar powered vehicle utilizing incident sol...
8,50662501,US,40942082,A,1982-08-19,1982,23620416,Y02T 90/12,Solar powered vehicle,A solar powered vehicle utilizing incident sol...
9,50662501,US,40942082,A,1982-08-19,1982,23620416,Y02T 90/14,Solar powered vehicle,A solar powered vehicle utilizing incident sol...


In [18]:
# Nettoyage et dédoublonnage par famille DOCDB
df_clean_abstracts = (
    df_cpc
    .dropna(subset=['appln_abstract'])
    .drop_duplicates(subset=['docdb_family_id'], keep='first')
)

print(f"Nombre de résumés uniques prêts pour le NLP : {len(df_clean_abstracts)}")
print(df_clean_abstracts)

Nombre de résumés uniques prêts pour le NLP : 2425
        appln_id appln_auth      appln_nr appln_kind appln_filing_date  \
0       50358722         US      3758836D         A         1972-02-03   
2       53746527         US      83979977         A         1977-10-06   
3       49511120         US      31139281         A         1981-10-14   
7       50662501         US      40942082         A         1982-08-19   
10      34948242         JP      24078586         A         1986-10-09   
...          ...        ...           ...        ...               ...   
12168  601907447         CN  202311341946         A         2023-10-17   
12169  601895331         CN  202311345056         A         2023-10-18   
12170  602456488         CN  202311396528         A         2023-10-26   
12171  604010064         CN  202311635314         A         2023-12-01   
12172  604867157         CN  202311719536         A         2023-12-14   

       appln_filing_year  docdb_family_id cpc_class_symbol  

In [21]:
pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 41.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993332 sha256=5a529e2012b29e37aad28ff287ebe74d73aca998a6ca65cece2858a2ba70dd20
  Stored in directory: /home/jovyan/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
Note: you may need to restart the kernel to use updated packages.


In [24]:
from langdetect import detect

def is_pure_english(text):
    if not isinstance(text, str) or len(text.strip()) < 30:
        return False
    if "??" in text: # Élimine le bruit d'encodage
        return False
    try:
        return detect(text) == 'en'
    except:
        return False

# Application du filtre
mask_english = df_clean_abstracts['appln_abstract'].apply(is_pure_english)

# Dataset propre pour le NLP
df_english_nlp = df_clean_abstracts[mask_english].copy()
#print(df_english_nlp)

# Dataset mis de côté
df_non_english = df_clean_abstracts[~mask_english].copy()
print(df_non_english)

        appln_id appln_auth      appln_nr appln_kind appln_filing_date  \
296       177052         EP      02013826         A         2002-06-21   
304     13028017         DE      10213792         A         2002-03-27   
985    273600939         DE  112008000424         T         2008-02-19   
1018   332613657         KR   20070015647         U         2007-09-19   
1389   315875979         DE  102008037574         A         2008-11-21   
...          ...        ...           ...        ...               ...   
11834  592602664         DE  202022102525         U         2022-05-09   
11837  581666766         DE  202022104413         U         2022-08-03   
11928  578156121         CN  202210518023         A         2022-05-13   
12171  604010064         CN  202311635314         A         2023-12-01   
12172  604867157         CN  202311719536         A         2023-12-14   

       appln_filing_year  docdb_family_id cpc_class_symbol  \
296                 2002         29716817      Y0

In [25]:
print(df_english_nlp)

        appln_id appln_auth      appln_nr appln_kind appln_filing_date  \
0       50358722         US      3758836D         A         1972-02-03   
2       53746527         US      83979977         A         1977-10-06   
3       49511120         US      31139281         A         1981-10-14   
7       50662501         US      40942082         A         1982-08-19   
10      34948242         JP      24078586         A         1986-10-09   
...          ...        ...           ...        ...               ...   
12166  601924036         CN  202311330483         A         2023-10-16   
12167  601915381         CN  202311337848         A         2023-10-17   
12168  601907447         CN  202311341946         A         2023-10-17   
12169  601895331         CN  202311345056         A         2023-10-18   
12170  602456488         CN  202311396528         A         2023-10-26   

       appln_filing_year  docdb_family_id cpc_class_symbol  \
0                   1972         22834888      Y0

In [27]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Mots vides standards + vocabulaire juridique/technique générique des brevets
PATENT_STOP_WORDS = list(ENGLISH_STOP_WORDS.union([
    "invention", "method", "system", "device", "apparatus", "plurality",
    "comprising", "configured", "related", "disclosed", "according",
    "embodiment", "wherein", "characterised", "claim", "claims", "module",
    "provided", "assembly", "unit", "portion", "means", "disposition"
]))

In [33]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

def extract_patent_keywords(df_clean, text_column='appln_abstract'):
    """
    Extrait les N-grammes (2 à 3 mots) les plus pertinents d'une colonne d'abstracts.
    """
    # 1. Configuration du vectoriseur TF-IDF
    tfidf = TfidfVectorizer(
        ngram_range=(2, 3),        # Bigrammes et Trigrammes
        stop_words=PATENT_STOP_WORDS,
        min_df=5,                  # L'expression doit apparaître dans au moins 5 brevets
        max_df=0.6                 # Élimine les termes présents dans plus de 60% des résumés
    )
    
    # 2. Application sur le corpus d'abstracts
    tfidf_matrix = tfidf.fit_transform(df_clean[text_column])
    
    # 3. Calcul du score moyen pour chaque mot-clé
    feature_names = tfidf.get_feature_names_out()
    scores = tfidf_matrix.sum(axis=0).A1
    
    # 4. Structuration dans un DataFrame Pandas
    df_keywords = pd.DataFrame({
        'keyword': feature_names,
        'importance_score': scores
    }).sort_values(by='importance_score', ascending=False).reset_index(drop=True)
    
    return df_keywords

# Utilisation sur votre dataset nettoyé
df_top_keywords = extract_patent_keywords(df_english_nlp)

# Affichage des 20 premiers résultats
print(df_top_keywords.head(20))

                      keyword  importance_score
0            electric vehicle         78.513672
1              energy storage         59.800112
2               charging pile         47.751136
3                power supply         45.114204
4            charging station         43.414927
5            power generation         38.299338
6                  power grid         32.698731
7              electric power         31.271439
8                   fuel cell         30.377988
9                  wind power         29.856027
10          electric vehicles         28.764513
11           vehicle charging         27.978590
12            electric energy         26.874782
13            storage battery         26.758259
14          electrical energy         26.334581
15               power source         24.759762
16           electrical power         22.539294
17  electric vehicle charging         21.696977
18               battery pack         21.348316
19            unmanned aerial         20

In [40]:
from pathlib import Path

# Définit le chemin et crée les dossiers parents si nécessaire
output_path = Path("extracted_keywords_tfidf.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

# Sauvegarde
df_top_keywords.to_csv(output_path, index=False, encoding="utf-8")

#df_top_keywords.to_csv("innovation-management-datascience26/tutorial_0/step_0/extracted_keywords_tfidf.csv", index=False)